# PerFedAvg Server

> AS implemented in https://proceedings.neurips.cc/paper/2020/file/24389bfe4fe2eba8bf9aa9203a44cdad-Paper.pdf

In [ ]:
#| default_exp servers.perfedavg

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
import os
from tqdm import tqdm
import numpy as np
import pandas as pd
from loguru import logger
from fastcore.utils import *
from fastcore.all import *

import torch
import torch.nn.functional as F

from fedai.servers.base_server import BaseServer
from fedai.utils.registery import AlgorithmRegistry

<torch._C.Generator>

## Per-FedAvg

Personalized Federated Learning with Theoretical Guarantees: A Model-Agnostic Meta-Learning Approach

In [ ]:
#| export
@AlgorithmRegistry.register_server("perfedavg")
class ServerPerFedAvg(BaseServer):
    def __init__(self,
                 cfg,
                 client_selector,
                 client_cls,
                 criterion,
                 fds,
                 writer= None,
                 device= None,
                 **kwargs
                 ):  
                 
        super().__init__(cfg, client_selector, client_cls, criterion, fds, writer, device, **kwargs) 

### Aggregation

In [ ]:
#| export
@patch
def aggregate(self: ServerPerFedAvg, lst_active_ids, comm_round, len_clients_ds):
    # m_t = sum(len_clients_ds.values())
    weight = self.cfg.m * self.cfg.num_clients
    with torch.no_grad():
        global_model = None
        
        for id in lst_active_ids:
            client_state_dict = self.state_mgr.get_state(id).get('model', None)

            if global_model is None:
                global_model = {k: torch.zeros_like(v) for k, v in client_state_dict.items()}

            # n_k = len_clients_ds[id]
            # weight = self.cfg.m * self.cfg.num_clients#n_k / m_t
            for key in global_model.keys():
                global_model[key].add_(client_state_dict[key], alpha=weight)

        self.model.load_state_dict(global_model)

        for id in lst_active_ids:
            self.state_mgr.set_state(id, {
                    'model': self.model.state_dict(),
                    'optimizer': self.state_mgr.get_state(id).get('optimizer', None),
            })
        

### Evaluation

In [ ]:
#| export
@patch
def evaluate(self: ServerPerFedAvg, t):
    self.cfg.agg ="mtl"
    lst_train_res = []
    lst_test_res = []
    for id in range(self.cfg.num_clients):

        client_state = self.state_mgr.get_state(id)
        client = self.client_fn(id= id, comm_round= t, client_state= client_state)
        client.train_step(num_steps= 1)
        
        res_train = client.evaluate_local(loader= 'train')
        lst_train_res.append(res_train)

        res_test = client.evaluate_local(loader= 'test')
        lst_test_res.append(res_test)
        
    self.cfg.agg = "one_model"
    return lst_train_res, lst_test_res    


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()